# House Prices - Advanced Regression Techniques

## Notebook 02: Feature Engineering

This notebook transforms the raw Ames Housing dataset into a model-ready dataset.

The objective is to apply the findings from the EDA notebook and prepare features that can improve predictive performance in the modeling stage.

The main tasks are:

1. Load the training, test, and sample submission files.
2. Preserve the target variable and identifiers.
3. Combine train and test data for consistent preprocessing.
4. Handle missing values using domain logic.
5. Engineer predictive features from area, age, bathroom, garage, basement, and quality variables.
6. Encode ordinal and nominal categorical variables.
7. Correct skewness in numerical variables.
8. Split the processed data back into train and test sets.
9. Save the processed outputs for modeling.

In [1]:
# ============================================================
# Imports and Configuration
# ============================================================

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import skew

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Load Competition Files

The competition provides:

- `train.csv`: training data with `SalePrice`
- `test.csv`: test data without `SalePrice`
- `sample_submission.csv`: required Kaggle submission format
- `data_description.txt`: variable descriptions

The feature engineering notebook uses both train and test data to ensure that all preprocessing steps create the same columns for both datasets.

In [2]:
# ============================================================
# File Paths
# ============================================================

BASE_PATH = Path("/kaggle/input/competitions/house-prices-advanced-regression-techniques")

TRAIN_PATH = BASE_PATH / "train.csv"
TEST_PATH = BASE_PATH / "test.csv"
SAMPLE_PATH = BASE_PATH / "sample_submission.csv"
DESC_PATH = BASE_PATH / "data_description.txt"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

Train shape: (1460, 81)
Test shape: (1459, 80)
Sample submission shape: (1459, 2)


In [3]:
train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0000,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0000,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,"2,003.0000",RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0000,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0000,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,"1,976.0000",RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0000,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0000,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,"2,001.0000",RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0000,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0000,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,"1,998.0000",Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0000,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0000,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,"2,000.0000",RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [4]:
test.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0000,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,6,1961,1961,Gable,CompShg,VinylSd,VinylSd,NaN,0.0000,TA,TA,CBlock,TA,TA,No,Rec,468.0000,LwQ,144.0000,270.0000,882.0000,GasA,TA,Y,SBrkr,896,0,0,896,0.0000,0.0000,1,0,2,1,TA,5,Typ,0,NaN,Attchd,"1,961.0000",Unf,1.0000,730.0000,TA,TA,Y,140,0,0,0,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0000,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,6,1958,1958,Hip,CompShg,Wd Sdng,Wd Sdng,BrkFace,108.0000,TA,TA,CBlock,TA,TA,No,ALQ,923.0000,Unf,0.0000,406.0000,"1,329.0000",GasA,TA,Y,SBrkr,1329,0,0,1329,0.0000,0.0000,1,1,3,1,Gd,6,Typ,0,NaN,Attchd,"1,958.0000",Unf,1.0000,312.0000,TA,TA,Y,393,36,0,0,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0000,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,5,5,1997,1998,Gable,CompShg,VinylSd,VinylSd,NaN,0.0000,TA,TA,PConc,Gd,TA,No,GLQ,791.0000,Unf,0.0000,137.0000,928.0000,GasA,Gd,Y,SBrkr,928,701,0,1629,0.0000,0.0000,2,1,3,1,TA,6,Typ,1,TA,Attchd,"1,997.0000",Fin,2.0000,482.0000,TA,TA,Y,212,34,0,0,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0000,9978,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,6,6,1998,1998,Gable,CompShg,VinylSd,VinylSd,BrkFace,20.0000,TA,TA,PConc,TA,TA,No,GLQ,602.0000,Unf,0.0000,324.0000,926.0000,GasA,Ex,Y,SBrkr,926,678,0,1604,0.0000,0.0000,2,1,3,1,Gd,7,Typ,1,Gd,Attchd,"1,998.0000",Fin,2.0000,470.0000,TA,TA,Y,360,36,0,0,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0000,5005,Pave,NaN,IR1,HLS,AllPub,Inside,Gtl,StoneBr,Norm,Norm,TwnhsE,1Story,8,5,1992,1992,Gable,CompShg,HdBoard,HdBoard,NaN,0.0000,Gd,TA,PConc,Gd,TA,No,ALQ,263.0000,Unf,0.0000,"1,017.0000","1,280.0000",GasA,Ex,Y,SBrkr,1280,0,0,1280,0.0000,0.0000,2,0,2,1,Gd,5,Typ,0,NaN,Attchd,"1,992.0000",RFn,2.0000,506.0000,TA,TA,Y,0,82,0,0,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


---
## 2. Preserve Identifiers and Target

`Id` is not a predictive variable, but it must be preserved for the final submission.

The target variable, `SalePrice`, is transformed using `log1p` because the EDA showed that the raw target is right-skewed. The Kaggle metric is based on logarithmic error, so modeling on the log-transformed target is appropriate.

In [5]:
# ============================================================
# Preserve IDs and Target
# ============================================================

target = "SalePrice"

train_ids = train["Id"].copy()
test_ids = test["Id"].copy()

y_train = np.log1p(train[target])

train_features = train.drop(columns=[target])
test_features = test.copy()

print("Training target shape:", y_train.shape)
print("Train features shape:", train_features.shape)
print("Test features shape:", test_features.shape)

Training target shape: (1460,)
Train features shape: (1460, 80)
Test features shape: (1459, 80)


In [6]:
pd.DataFrame({
    "raw_saleprice": train[target],
    "log_saleprice": y_train
}).head()

,raw_saleprice,log_saleprice
0,208500,12.2477
1,181500,12.1090
2,223500,12.3172
3,140000,11.8494
4,250000,12.4292


---
## 3. Combine Train and Test for Consistent Processing

Train and test data are combined before preprocessing so that categorical encoding creates the same set of columns for both datasets.

This prevents a common problem where a category appears in the test data but not in the training data, or vice versa.

In [7]:
# ============================================================
# Combine Train and Test
# ============================================================

n_train = train_features.shape[0]
n_test = test_features.shape[0]

combined = pd.concat(
    [train_features, test_features],
    axis=0,
    ignore_index=True
)

print("Combined shape:", combined.shape)
combined.head()

Combined shape: (2919, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,60,RL,65.0000,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0000,Gd,TA,PConc,Gd,TA,No,GLQ,706.0000,Unf,0.0000,150.0000,856.0000,GasA,Ex,Y,SBrkr,856,854,0,1710,1.0000,0.0000,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,"2,003.0000",RFn,2.0000,548.0000,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,2,20,RL,80.0000,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0000,TA,TA,CBlock,Gd,TA,Gd,ALQ,978.0000,Unf,0.0000,284.0000,"1,262.0000",GasA,Ex,Y,SBrkr,1262,0,0,1262,0.0000,1.0000,2,0,3,1,TA,6,Typ,1,TA,Attchd,"1,976.0000",RFn,2.0000,460.0000,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,3,60,RL,68.0000,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0000,Gd,TA,PConc,Gd,TA,Mn,GLQ,486.0000,Unf,0.0000,434.0000,920.0000,GasA,Ex,Y,SBrkr,920,866,0,1786,1.0000,0.0000,2,1,3,1,Gd,6,Typ,1,TA,Attchd,"2,001.0000",RFn,2.0000,608.0000,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,4,70,RL,60.0000,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0000,TA,TA,BrkTil,TA,Gd,No,ALQ,216.0000,Unf,0.0000,540.0000,756.0000,GasA,Gd,Y,SBrkr,961,756,0,1717,1.0000,0.0000,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,"1,998.0000",Unf,3.0000,642.0000,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,5,60,RL,84.0000,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0000,Gd,TA,PConc,Gd,TA,Av,GLQ,655.0000,Unf,0.0000,490.0000,"1,145.0000",GasA,Ex,Y,SBrkr,1145,1053,0,2198,1.0000,0.0000,2,1,4,1,Gd,9,Typ,1,TA,Attchd,"2,000.0000",RFn,3.0000,836.0000,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal


---
## 4. Initial Feature Type Review

Before imputing missing values, we separate numerical and categorical features.

This helps guide the imputation strategy:

- Categorical missing values may mean absence of a feature.
- Numerical missing values may represent zero area, zero cars, or a true unknown value.

In [8]:
# ============================================================
# Feature Type Review
# ============================================================

numeric_cols = combined.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = combined.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

Numeric columns: 37
Categorical columns: 43


In [9]:
missing_summary = (
    combined.isna()
    .sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_summary["missing_percent"] = missing_summary["missing_count"] / len(combined) * 100
missing_summary

,missing_count,missing_percent
PoolQC,2909,99.6574
MiscFeature,2814,96.4029
Alley,2721,93.2169
Fence,2348,80.4385
MasVnrType,1766,60.5002
FireplaceQu,1420,48.6468
LotFrontage,486,16.6495
GarageQual,159,5.4471
GarageYrBlt,159,5.4471
GarageCond,159,5.4471


---
## 5. Missing Value Strategy

The Ames Housing dataset contains missing values with different meanings.

Some missing values represent true absence. For example:

- `PoolQC` missing usually means no pool.
- `Fence` missing usually means no fence.
- Garage-related missing values often mean no garage.
- Basement-related missing values often mean no basement.

Other missing values require imputation because they represent unavailable information rather than absence.

### 5.1 Categorical Missing Values That Mean Absence

These variables describe optional property features. Missing values are replaced with `"None"` because the property does not have that feature.

In [10]:
# ============================================================
# Categorical Absence Features
# ============================================================

none_cols = [
    "Alley",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "PoolQC",
    "Fence",
    "MiscFeature",
    "MasVnrType"
]

for col in none_cols:
    if col in combined.columns:
        combined[col] = combined[col].fillna("None")

print("Categorical absence values filled.")

Categorical absence values filled.


### 5.2 Numerical Missing Values That Mean Absence

For area, count, and year variables connected to optional structures, missing values are filled with `0`.

Examples:

- No garage means `GarageCars = 0`, `GarageArea = 0`, and `GarageYrBlt = 0`.
- No basement means basement area and basement bathroom counts are `0`.
- No masonry veneer means `MasVnrArea = 0`.

In [11]:
# ============================================================
# Numerical Absence Features
# ============================================================

zero_cols = [
    "GarageYrBlt",
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "BsmtFullBath",
    "BsmtHalfBath",
    "MasVnrArea"
]

for col in zero_cols:
    if col in combined.columns:
        combined[col] = combined[col].fillna(0)

print("Numerical absence values filled.")

Numerical absence values filled.


### 5.3 LotFrontage Imputation by Neighborhood

`LotFrontage` measures the linear feet of street connected to the property.

Because street frontage is strongly related to location and lot layout, it is better to impute missing values using the median `LotFrontage` within each `Neighborhood`, instead of using a single global median.

In [12]:
# ============================================================
# LotFrontage Imputation
# ============================================================

if "LotFrontage" in combined.columns and "Neighborhood" in combined.columns:
    combined["LotFrontage"] = combined.groupby("Neighborhood")["LotFrontage"].transform(
        lambda x: x.fillna(x.median())
    )

    combined["LotFrontage"] = combined["LotFrontage"].fillna(combined["LotFrontage"].median())

print("LotFrontage imputation complete.")

LotFrontage imputation complete.


### 5.4 Remaining Categorical Missing Values

Remaining categorical missing values are filled using the mode.

This is suitable for small numbers of missing values in variables such as zoning, exterior type, electrical system, kitchen quality, functionality, sale type, and utilities.

In [13]:
# ============================================================
# Remaining Categorical Imputation
# ============================================================

categorical_cols = combined.select_dtypes(include=["object"]).columns.tolist()

for col in categorical_cols:
    if combined[col].isna().sum() > 0:
        combined[col] = combined[col].fillna(combined[col].mode()[0])

print("Remaining categorical missing values filled.")

Remaining categorical missing values filled.


### 5.5 Remaining Numerical Missing Values

Remaining numerical missing values are filled using the median.

Median imputation is more robust than mean imputation because many housing variables are skewed.

In [14]:
# ============================================================
# Remaining Numerical Imputation
# ============================================================

numeric_cols = combined.select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_cols:
    if combined[col].isna().sum() > 0:
        combined[col] = combined[col].fillna(combined[col].median())

print("Remaining numerical missing values filled.")

Remaining numerical missing values filled.


In [15]:
# Final missing value check
remaining_missing = combined.isna().sum().sum()
print("Remaining missing values:", remaining_missing)

Remaining missing values: 0


---
## 6. Domain-Informed Feature Engineering

The EDA notebook showed that price is strongly influenced by:

- Total living area
- Basement size
- Garage capacity
- Number of bathrooms
- House age
- Remodeling status
- Neighborhood
- Quality ratings
- Presence of optional features such as garages, basements, fireplaces, and pools

This section creates features that capture those signals more directly.

### 6.1 Area Features

Area-related variables are among the strongest predictors of `SalePrice`.

Instead of using only individual square-footage columns, we create combined area features that better represent total usable space.

In [16]:
# ============================================================
# Area Features
# ============================================================

combined["TotalHouseSF"] = (
    combined["TotalBsmtSF"] +
    combined["1stFlrSF"] +
    combined["2ndFlrSF"]
)

combined["TotalLivingSF"] = (
    combined["GrLivArea"] +
    combined["TotalBsmtSF"]
)

combined["TotalFinishedSF"] = (
    combined["GrLivArea"] +
    combined["BsmtFinSF1"] +
    combined["BsmtFinSF2"]
)

combined["TotalPorchSF"] = (
    combined["OpenPorchSF"] +
    combined["EnclosedPorch"] +
    combined["3SsnPorch"] +
    combined["ScreenPorch"]
)

combined["TotalOutdoorSF"] = (
    combined["WoodDeckSF"] +
    combined["OpenPorchSF"] +
    combined["EnclosedPorch"] +
    combined["3SsnPorch"] +
    combined["ScreenPorch"] +
    combined["PoolArea"]
)

print("Area features created.")

Area features created.


### 6.2 Bathroom Features

Bathrooms are count-based features, but full and half bathrooms do not contribute equally.

A combined bathroom feature is created by counting half bathrooms as `0.5`.

In [17]:
# ============================================================
# Bathroom Features
# ============================================================

combined["TotalBathrooms"] = (
    combined["FullBath"] +
    0.5 * combined["HalfBath"] +
    combined["BsmtFullBath"] +
    0.5 * combined["BsmtHalfBath"]
)

combined["TotalFullBaths"] = (
    combined["FullBath"] +
    combined["BsmtFullBath"]
)

combined["TotalHalfBaths"] = (
    combined["HalfBath"] +
    combined["BsmtHalfBath"]
)

print("Bathroom features created.")

Bathroom features created.


### 6.3 Age and Remodeling Features

Raw year variables are useful, but age-based variables are easier for models to interpret.

This section creates:

- `HouseAge`: age of the house at sale
- `RemodAge`: years since remodel at sale
- `GarageAge`: age of garage at sale
- `IsRemodeled`: whether the home was remodeled
- `IsNewHouse`: whether the house was sold in the same year it was built

In [18]:
# ============================================================
# Age and Remodeling Features
# ============================================================

combined["HouseAge"] = combined["YrSold"] - combined["YearBuilt"]
combined["RemodAge"] = combined["YrSold"] - combined["YearRemodAdd"]
combined["GarageAge"] = combined["YrSold"] - combined["GarageYrBlt"]

combined.loc[combined["GarageYrBlt"] == 0, "GarageAge"] = 0

combined["IsRemodeled"] = (combined["YearBuilt"] != combined["YearRemodAdd"]).astype(int)
combined["IsNewHouse"] = (combined["YearBuilt"] == combined["YrSold"]).astype(int)

# Protect against rare negative age values
for col in ["HouseAge", "RemodAge", "GarageAge"]:
    combined[col] = combined[col].clip(lower=0)

print("Age and remodeling features created.")

Age and remodeling features created.


### 6.4 Binary Presence Indicators

Many optional property features have strong price effects.

This section creates binary indicators for whether the property has:

- Garage
- Basement
- Fireplace
- Pool
- Fence
- Alley access
- Masonry veneer
- Deck
- Porch
- Second floor

In [19]:
# ============================================================
# Binary Presence Features
# ============================================================

combined["HasGarage"] = (combined["GarageArea"] > 0).astype(int)
combined["HasBasement"] = (combined["TotalBsmtSF"] > 0).astype(int)
combined["HasFireplace"] = (combined["Fireplaces"] > 0).astype(int)
combined["HasPool"] = (combined["PoolArea"] > 0).astype(int)
combined["HasFence"] = (combined["Fence"] != "None").astype(int)
combined["HasAlley"] = (combined["Alley"] != "None").astype(int)
combined["HasMasVnr"] = (combined["MasVnrArea"] > 0).astype(int)
combined["HasDeck"] = (combined["WoodDeckSF"] > 0).astype(int)
combined["HasPorch"] = (combined["TotalPorchSF"] > 0).astype(int)
combined["HasSecondFloor"] = (combined["2ndFlrSF"] > 0).astype(int)

print("Binary presence features created.")

Binary presence features created.


### 6.5 Quality and Size Interaction Features

The EDA showed that `OverallQual` and area features are among the strongest price drivers.

A large house with low quality is not equivalent to a large house with high quality. Interaction features help capture this relationship.

In [20]:
# ============================================================
# Interaction Features
# ============================================================

combined["OverallQual_TotalHouseSF"] = combined["OverallQual"] * combined["TotalHouseSF"]
combined["OverallQual_GrLivArea"] = combined["OverallQual"] * combined["GrLivArea"]
combined["OverallQual_GarageArea"] = combined["OverallQual"] * combined["GarageArea"]
combined["OverallQual_TotalBathrooms"] = combined["OverallQual"] * combined["TotalBathrooms"]

combined["GarageCars_GarageArea"] = combined["GarageCars"] * combined["GarageArea"]
combined["TotalHouseSF_TotalBathrooms"] = combined["TotalHouseSF"] * combined["TotalBathrooms"]

print("Interaction features created.")

Interaction features created.


### 6.6 Simplified Overall Property Score

This section creates a compact quality score using several quality and condition variables.

The score is not meant to replace individual variables. It gives models an additional summary feature representing overall property strength.

In [21]:
# ============================================================
# Overall Property Score
# ============================================================

combined["OverallScore"] = (
    combined["OverallQual"] +
    combined["OverallCond"]
)

combined["QualityAgeInteraction"] = combined["OverallQual"] * (combined["HouseAge"] + 1)

print("Overall score features created.")

Overall score features created.


---
## 7. Ordinal Encoding

Some categorical variables have a natural order.

For example:

- `Ex` is better than `Gd`
- `Gd` is better than `TA`
- `TA` is better than `Fa`
- `Fa` is better than `Po`

These should be ordinal-encoded rather than one-hot encoded because their order carries meaning.

In [22]:
# ============================================================
# Ordinal Encoding
# ============================================================

quality_map = {
    "None": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
}

quality_cols = [
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "HeatingQC",
    "KitchenQual",
    "FireplaceQu",
    "GarageQual",
    "GarageCond",
    "PoolQC"
]

for col in quality_cols:
    if col in combined.columns:
        combined[col] = combined[col].map(quality_map).fillna(0).astype(int)

print("Quality variables ordinal-encoded.")

Quality variables ordinal-encoded.


### 7.1 Additional Ordinal Encodings

Other variables also contain ordered categories, including basement exposure, basement finish, garage finish, paved driveway, land slope, lot shape, and functionality.

These are encoded manually so their order is preserved.

In [23]:
# ============================================================
# Additional Ordinal Encoding
# ============================================================

ordinal_maps = {
    "BsmtExposure": {
        "None": 0,
        "No": 1,
        "Mn": 2,
        "Av": 3,
        "Gd": 4
    },
    "BsmtFinType1": {
        "None": 0,
        "Unf": 1,
        "LwQ": 2,
        "Rec": 3,
        "BLQ": 4,
        "ALQ": 5,
        "GLQ": 6
    },
    "BsmtFinType2": {
        "None": 0,
        "Unf": 1,
        "LwQ": 2,
        "Rec": 3,
        "BLQ": 4,
        "ALQ": 5,
        "GLQ": 6
    },
    "GarageFinish": {
        "None": 0,
        "Unf": 1,
        "RFn": 2,
        "Fin": 3
    },
    "PavedDrive": {
        "N": 0,
        "P": 1,
        "Y": 2
    },
    "LandSlope": {
        "Gtl": 0,
        "Mod": 1,
        "Sev": 2
    },
    "LotShape": {
        "Reg": 0,
        "IR1": 1,
        "IR2": 2,
        "IR3": 3
    },
    "Functional": {
        "Sal": 0,
        "Sev": 1,
        "Maj2": 2,
        "Maj1": 3,
        "Mod": 4,
        "Min2": 5,
        "Min1": 6,
        "Typ": 7
    },
    "Utilities": {
        "ELO": 0,
        "NoSeWa": 1,
        "NoSewr": 2,
        "AllPub": 3
    },
    "CentralAir": {
        "N": 0,
        "Y": 1
    }
}

for col, mapping in ordinal_maps.items():
    if col in combined.columns:
        combined[col] = combined[col].map(mapping).fillna(0).astype(int)

print("Additional ordinal variables encoded.")

Additional ordinal variables encoded.


---
## 8. Convert Numeric-Looking Categorical Variables

Some variables are stored as numbers but represent categories rather than continuous measurements.

`MSSubClass` is a building class code, not a continuous numerical feature. It should be treated as categorical.

In [24]:
# ============================================================
# Numeric-Looking Categorical Features
# ============================================================

for col in ["MSSubClass"]:
    if col in combined.columns:
        combined[col] = combined[col].astype(str)

print("Numeric-looking categorical variables converted.")

Numeric-looking categorical variables converted.


---
## 9. Nominal Encoding

After ordinal variables have been manually encoded, the remaining categorical variables are nominal variables with no natural order.

These are one-hot encoded using `pd.get_dummies`.

In [25]:
# ============================================================
# One-Hot Encoding
# ============================================================

combined_encoded = pd.get_dummies(combined, drop_first=False)

print("Shape before encoding:", combined.shape)
print("Shape after encoding:", combined_encoded.shape)

Shape before encoding: (2919, 111)
Shape after encoding: (2919, 275)


In [26]:
combined_encoded.head()

,Id,LotFrontage,LotArea,LotShape,Utilities,LandSlope,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,ExterCond,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,HeatingQC,CentralAir,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,MiscVal,MoSold,YrSold,TotalHouseSF,TotalLivingSF,TotalFinishedSF,TotalPorchSF,TotalOutdoorSF,TotalBathrooms,TotalFullBaths,TotalHalfBaths,HouseAge,RemodAge,GarageAge,IsRemodeled,IsNewHouse,HasGarage,HasBasement,HasFireplace,HasPool,HasFence,HasAlley,...,Exterior1st_Plywood,Exterior1st_Stone,Exterior1st_Stucco,Exterior1st_VinylSd,Exterior1st_Wd Sdng,Exterior1st_WdShing,Exterior2nd_AsbShng,Exterior2nd_AsphShn,Exterior2nd_Brk Cmn,Exterior2nd_BrkFace,Exterior2nd_CBlock,Exterior2nd_CmentBd,Exterior2nd_HdBoard,Exterior2nd_ImStucc,Exterior2nd_MetalSd,Exterior2nd_Other,Exterior2nd_Plywood,Exterior2nd_Stone,Exterior2nd_Stucco,Exterior2nd_VinylSd,Exterior2nd_Wd Sdng,Exterior2nd_Wd Shng,MasVnrType_BrkCmn,MasVnrType_BrkFace,MasVnrType_None,MasVnrType_Stone,Foundation_BrkTil,Foundation_CBlock,Foundation_PConc,Foundation_Slab,Foundation_Stone,Foundation_Wood,Heating_Floor,Heating_GasA,Heating_GasW,Heating_Grav,Heating_OthW,Heating_Wall,Electrical_FuseA,Electrical_FuseF,Electrical_FuseP,Electrical_Mix,Electrical_SBrkr,GarageType_2Types,GarageType_Attchd,GarageType_Basment,GarageType_BuiltIn,GarageType_CarPort,GarageType_Detchd,GarageType_None,Fence_GdPrv,Fence_GdWo,Fence_MnPrv,Fence_MnWw,Fence_None,MiscFeature_Gar2,MiscFeature_None,MiscFeature_Othr,MiscFeature_Shed,MiscFeature_TenC,SaleType_COD,SaleType_CWD,SaleType_Con,SaleType_ConLD,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,1,65.0000,8450,0,3,0,7,5,2003,2003,196.0000,4,3,4,3,1,6,706.0000,1,0.0000,150.0000,856.0000,5,1,856,854,0,1710,1.0000,0.0000,2,1,3,1,4,8,7,0,0,"2,003.0000",2,2.0000,548.0000,3,3,2,0,61,0,0,0,0,0,0,2,2008,"2,566.0000","2,566.0000","2,416.0000",61,61,3.5000,3.0000,1.0000,5,5,5.0000,0,0,1,1,0,0,0,0,...,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False
1,2,80.0000,9600,0,3,0,6,8,1976,1976,0.0000,3,3,4,3,4,5,978.0000,1,0.0000,284.0000,"1,262.0000",5,1,1262,0,0,1262,0.0000,1.0000,2,0,3,1,3,6,7,1,3,"1,976.0000",2,2.0000,460.0000,3,3,2,298,0,0,0,0,0,0,0,5,2007,"2,524.0000","2,524.0000","2,240.0000",0,298,2.5000,2.0000,1.0000,31,31,31.0000,0,0,1,1,1,0,0,0,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False
2,3,68.0000,11250,1,3,0,7,5,2001,2002,162.0000,4,3,4,3,2,6,486.0000,1,0.0000,434.0000,920.0000,5,1,920,866,0,1786,1.0000,0.0000,2,1,3,1,4,6,7,1,3,"2,001.0000",2,2.0000,608.0000,3,3,2,0,42,0,0,0,0,0,0,9,2008,"2,706.0000","2,706.0000","2,272.0000",42,42,3.5000,3.0000,1.0000,7,6,7.0000,1,0,1,1,1,0,0,0,...,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,

---
## 10. Skewness Correction

Many housing variables are right-skewed. Examples include lot area, basement area, porch area, and miscellaneous values.

For numerical features with strong positive skewness, `log1p` transformation is applied.

This transformation is only applied to non-binary numerical features.

In [27]:
# ============================================================
# Skewness Correction
# ============================================================

numeric_features = combined_encoded.select_dtypes(include=[np.number]).columns.tolist()

binary_features = [
    col for col in numeric_features
    if combined_encoded[col].nunique() <= 2
]

skew_candidates = [
    col for col in numeric_features
    if col not in binary_features and col != "Id"
]

skewness = combined_encoded[skew_candidates].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)

high_skew = skewness[skewness > 0.75]

print("Number of skewed features:", len(high_skew))
high_skew.head(30)

Number of skewed features: 41


MiscVal                       21.9472
PoolQC                        18.4091
PoolArea                      16.8983
LotArea                       12.8224
LowQualFinSF                  12.0888
3SsnPorch                     11.3761
LandSlope                      4.9752
KitchenAbvGr                   4.3023
BsmtFinSF2                     4.1461
EnclosedPorch                  4.0039
ScreenPorch                    3.9467
BsmtHalfBath                   3.9316
BsmtFinType2                   3.1523
MasVnrArea                     2.6136
OpenPorchSF                    2.5351
TotalPorchSF                   2.2373
TotalHouseSF_TotalBathrooms    2.2210
OverallQual_TotalHouseSF       2.1423
TotalOutdoorSF                 1.9015
WoodDeckSF                     1.8424
TotalFinishedSF                1.8407
OverallQual_GrLivArea          1.8091
TotalHouseSF                   1.5115
TotalLivingSF                  1.5112
LotFrontage                    1.5057
1stFlrSF                       1.4696
BsmtFinSF1  

In [28]:
# Apply log1p transformation to highly skewed features
for col in high_skew.index:
    combined_encoded[col] = np.log1p(combined_encoded[col])

print("Skewness correction complete.")

Skewness correction complete.


---
## 11. Final Train-Test Split

After preprocessing and feature engineering, the combined dataset is split back into training and test sets.

The training set will be used in the modeling notebook. The test set will be used to generate the Kaggle submission.

In [29]:
# ============================================================
# Final Split
# ============================================================

X_train = combined_encoded.iloc[:n_train, :].copy()
X_test = combined_encoded.iloc[n_train:, :].copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)

X_train shape: (1460, 275)
X_test shape: (1459, 275)
y_train shape: (1460,)


In [30]:
# Verify alignment
assert X_train.shape[0] == len(y_train)
assert X_test.shape[0] == len(test_ids)

print("Train-test split verified.")

Train-test split verified.


---
## 12. Final Data Quality Checks

Before saving the processed data, we check:

- Missing values
- Infinite values
- Duplicate columns
- Shape consistency

In [31]:
# ============================================================
# Final Data Checks
# ============================================================

print("Missing values in X_train:", X_train.isna().sum().sum())
print("Missing values in X_test:", X_test.isna().sum().sum())

print("Infinite values in X_train:", np.isinf(X_train.select_dtypes(include=[np.number])).sum().sum())
print("Infinite values in X_test:", np.isinf(X_test.select_dtypes(include=[np.number])).sum().sum())

print("Duplicate columns:", X_train.columns.duplicated().sum())

Missing values in X_train: 0
Missing values in X_test: 0
Infinite values in X_train: 0
Infinite values in X_test: 0
Duplicate columns: 0


In [32]:
# Replace any rare infinite values, just in case
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())

print("Final missing values in X_train:", X_train.isna().sum().sum())
print("Final missing values in X_test:", X_test.isna().sum().sum())

Final missing values in X_train: 0
Final missing values in X_test: 0


---
## 13. Save Processed Outputs

The processed datasets are saved to `/kaggle/working`.

These files will be used in the modeling notebook:

- `X_train_fe.csv`
- `X_test_fe.csv`
- `y_train_log.csv`
- `train_ids.csv`
- `test_ids.csv`

In [33]:
# ============================================================
# Save Outputs
# ============================================================

OUTPUT_DIR = Path("/kaggle/working/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_train.to_csv(OUTPUT_DIR / "X_train_fe.csv", index=False)
X_test.to_csv(OUTPUT_DIR / "X_test_fe.csv", index=False)
pd.Series(y_train, name="SalePriceLog").to_csv(OUTPUT_DIR / "y_train_log.csv", index=False)
pd.Series(train_ids, name="Id").to_csv(OUTPUT_DIR / "train_ids.csv", index=False)
pd.Series(test_ids, name="Id").to_csv(OUTPUT_DIR / "test_ids.csv", index=False)

print("Processed files saved to:", OUTPUT_DIR)

Processed files saved to: /kaggle/working/processed


In [34]:
os.listdir(OUTPUT_DIR)

['X_train_fe.csv',
 'y_train_log.csv',
 'test_ids.csv',
 'train_ids.csv',
 'X_test_fe.csv']

---
## 14. Feature Engineering Summary

This notebook transformed the raw Ames Housing dataset into a model-ready format.

Key steps completed:

1. `SalePrice` was transformed using `log1p` based on the right-skew observed in EDA.
2. Train and test data were combined to ensure consistent preprocessing.
3. Missing values were handled using domain-specific logic.
4. Absence-related categorical values were filled with `"None"`.
5. Absence-related numerical values were filled with `0`.
6. `LotFrontage` was imputed using neighborhood-level medians.
7. Area, bathroom, age, remodeling, porch, outdoor, and binary presence features were engineered.
8. Quality and condition variables were ordinal-encoded.
9. Remaining nominal categorical variables were one-hot encoded.
10. Highly skewed non-binary numerical features were transformed using `log1p`.
11. Processed train and test matrices were saved for modeling.

The next notebook should focus on model training, cross-validation, hyperparameter tuning, model comparison, and final Kaggle submission generation.

# End of Notebook 02

Next notebook:

`03_modeling_and_optimization.ipynb`